# 03. Reduzir variância

Randomizar deixa a estimativa **não enviesada**, mas não necessariamente **precisa**. Se você mede covariáveis correlacionadas com o resultado, dá para usá-las para encolher o erro-padrão sem introduzir viés. Dois caminhos na biblioteca: `LinEstimator` (ajuste por regressão) e `CUPED` (uso de um período pré-experimento).

## O experimento

O resultado é fortemente explicado por uma covariável `x` e por uma medida pré-experimento `y_pre`. O efeito verdadeiro é `0.5`.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD

rng = np.random.default_rng(11)
n = 200
x = rng.normal(size=n)
y_pre = rng.normal(size=n)
base = 1.5 * x + 1.5 * y_pre + rng.normal(size=n) * 0.5   # baseline explicada por x e y_pre
tau = 0.5
y0 = base
y1 = base + tau

df = pd.DataFrame({"x": x, "y_pre": y_pre})
design = CRD(p=0.5, seed=11)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=11
)

## Mesma estimativa, precisão diferente

Os três estimadores miram o mesmo ATE. Usamos o `BootstrapCI` (que aceita qualquer estimador escalar) para comparar o **erro-padrão**. Lin e CUPED devem ter SE menor que a diferença de médias simples.

In [ ]:
from skxperiments.estimators.cuped import CUPED
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.estimators.lin_estimator import LinEstimator
from skxperiments.inference import BootstrapCI

estimators = {
    "DifferenceInMeans": DifferenceInMeans(outcome_col="y"),
    "Lin (x)": LinEstimator(outcome_col="y", covariates=["x"]),
    "CUPED (y_pre)": CUPED(outcome_col="y", pre_experiment_col="y_pre"),
}
rows = []
for name, est in estimators.items():
    res = BootstrapCI(
        estimator=est, method="percentile", n_resamples=800, seed=0
    ).fit(assignment).estimate()
    rows.append({"estimador": name, "ATE": res.ate, "SE": res.se})
pd.DataFrame(rows)

## O que aprendemos

- Covariáveis correlacionadas com o resultado **reduzem a variância** da estimativa, mantendo a ausência de viés.
- `LinEstimator` ajusta por covariáveis medidas no experimento; `CUPED` usa uma medida pré-experimento.
- SE menor significa IC mais estreito e mais poder com o mesmo tamanho de amostra.

**Próximo:** `04. Equilíbrio e rerandomização` mostra como checar e melhorar o balanço das covariáveis.